In [ ]:
# Import Libraries

import os
import glob
import random
from dotenv import load_dotenv
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
import gradio as gr

C:\Users\Acer\AppData\Local\Temp\ipykernel_18908\58097278.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader
e:\Vehicle_RAG_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv(override=True)

KNOWLEDGE_BASE = "vehicle_knowledge_base"
DB_NAME = "vector_db"

In [ ]:
# STAGE 1

folders = glob.glob(f"{KNOWLEDGE_BASE}/*")
documents = []

for folder in folders:
    category = os.path.basename(folder)
    loader = DirectoryLoader(
        folder,
        glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"}
    )

    folder_docs = loader.load()

    for doc in folder_docs:
        source_path = os.path.relpath(doc.metadata["source"], KNOWLEDGE_BASE)
        source_path = source_path.replace("\\", "/") 

        doc.metadata["source"] = source_path
        doc.metadata["category"] = category
        documents.append(doc)

print(f"Total documents loaded: {len(documents)}")

Total documents loaded: 15


In [ ]:
# STAGE 2

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)}")

Total chunks created: 66


In [ ]:
# STAGE 3

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

if os.path.exists(DB_NAME):
    print("Loading existing database...")
    
    vector_store = Chroma(
        persist_directory=DB_NAME,
        embedding_function=embeddings
    )

else:
    print("Creating vector database...")

    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=DB_NAME,
    )

print(f"Vectors in database: {vector_store._collection.count()}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1175.91it/s]


Creating vector database...
Vectors in database: 66


In [ ]:
# STAGE 4

def retrieve_from_domain(query: str, category: str, k: int = 3) -> str:
    """Helper extraction function to execute precise metadata-filtered routing."""
    docs = vector_store.similarity_search(query, k=k, filter={"category": category})
    context = ""
    for doc in docs:
        context += f"\n[SOURCE: {doc.metadata['source']}]\n {doc.page_content}\n"
    return context

@tool
def query_fleet_catalog(query: str) -> str:
    """Useful for searching technical specifications, model variants, fuel economy and starting MSRP prices of vehicles."""
    return retrieve_from_domain(query, "fleet_catalog")

@tool
def query_sales_and_finance(query: str) -> str:
    """Useful for answering purchase steps, financing rates, trade in rules and active discounts rates."""
    return retrieve_from_domain(query, "sales_and_finance")

@tool
def query_rental_operations(query: str) -> str:
    """Useful for retrieving rental age limits, rental rates, insurance requirements and driver rules."""
    return retrieve_from_domain(query, "rental_operations")

@tool
def query_service_and_warranty(query: str) -> str:
    """Useful for answering active warranties, routine maintenance schedules, pricing and coverage details."""
    return retrieve_from_domain(query, "service_and_warranty")

@tool
def query_company_information(query: str) -> str:
    """Queries the dealership's physical showroom locations, contact numbers and operating hours ."""
    return retrieve_from_domain(query, "company_information")

tools = [query_fleet_catalog, query_sales_and_finance, query_rental_operations, query_service_and_warranty, query_company_information]

In [ ]:
# STAGE 5

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

SYSTEM_PROMPT = """You are an elite, helpful virtual customer assistant for 'Velocity Dealerships'.
your primary job is to provide accurate answers about vehicle models, financing policies, buying rules and rental agreements.
your goal is to guide the user seamlessly through pre qulification, sales, rentals, warranty claims and showroom info.

You have access to specific context-retrieval tools. you must query them to retrieve precise information.
If a customer's question requires information from multipe categories, make multiple tool calls.

RULES:
1. ONLY use information retrieved via your tools. Never use outside knowledge.
2. If the tools do not return relevant facts, say "I don't have that specific information"
3. Every response must reference the source document exactly where the information came from (e.g., [SOURCE: fleet_catalog/sedan_models.md]).
"""


In [8]:
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="chat_history"), ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)